<a href="https://colab.research.google.com/github/laurenpdog/Senior-Thesis/blob/main/Full_thesis_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thesis code for my bioinformatic workflow. My lab notebook usually links to the orginal files of code but this is the full list of all code.

After copying and pasting the modomics table in a sheet and inputting it into this code. This code will then parse uniprot for each pfam ID and records all accession IDs and taxanomy info. Took about 50 hours to run.

In [ ]:
import pandas as pd
from Bio import ExPASy
from Bio import SwissProt
import requests
import time
import pickle
import os


INPUT_EXCEL_FILE = "RNA Modification in Viruses.xlsx"
PFAM_ID_COLUMN_NAME = "Pfam ID"
OUTPUT_EXCEL_FILE = "Pfam_UniProt_Taxonomy_Data.xlsx"
GOOGLE_DRIVE_MOUNT_PATH = "/content/drive/MyDrive"
PICKLE_DIR = os.path.join(GOOGLE_DRIVE_MOUNT_PATH, "Pfam Pickles")

# Base URL for UniProt search
UNIPROT_BASE_URL = "https://rest.uniprot.org/uniprotkb/search"

# Methylase skipping
METHYLASE_PFAM_IDS_TO_SKIP = []
METHYLASE_KEYWORDS = ["methyltransferase", "methylase", "m6a", "m1g", "m5c", "rrmj", "trm"]

#uploading pfam IDs from excel
def load_pfam_ids(excel_file, column_name):
    print(f"Reading PFAM IDs from '{excel_file}'...")
    try:
        df_input = pd.read_excel(excel_file)
    except FileNotFoundError:
        print(f"Error: Input file '{excel_file}' not found. Please ensure it's in the correct directory.")
        return []
    except pd.errors.EmptyDataError:
        print(f"Error: '{excel_file}' is empty.")
        return []

    if column_name not in df_input.columns:
        raise ValueError(f"Column '{column_name}' not found in '{excel_file}'. "
                         "Please check the column name in your Excel file.")

    pfam_ids = df_input[column_name].dropna().unique().tolist()
    print(f"Found {len(pfam_ids)} unique PFAM IDs to process.")
    return [str(pid).strip() for pid in pfam_ids] # Ensure IDs are stripped strings for consistency

#pickle maker, to save all pickles to drive folder named Pfam Pickles
def save_to_pickle(data, pfam_id, directory):
    try:
        os.makedirs(directory, exist_ok=True)
        pickle_file_path = os.path.join(directory, f"pfam_{pfam_id}.pkl")
        with open(pickle_file_path, 'wb') as f:
            pickle.dump(data, f)
        print(f"  Data for {pfam_id} saved to {pickle_file_path}")
    except Exception as e:
        print(f"  Error saving pickle file for {pfam_id}: {e}")

#function to load pickles and check if there is already saved data, if not then fetch new ones
def load_from_pickle(pfam_id, directory):
    pickle_file_path = os.path.join(directory, f"pfam_{pfam_id}.pkl")

    # check if the file path exists and is a non empty file
    if os.path.exists(pickle_file_path) and os.path.getsize(pickle_file_path) > 0:
        print(f"  Cache found for {pfam_id}. Attempting to load from {pickle_file_path}...")
        try:
            with open(pickle_file_path, 'rb') as f:
                return pickle.load(f)
        except (EOFError, pickle.UnpicklingError) as e:
            # This handles cases where the file is empty or corrupted.
            print(f"  Error loading from corrupted/empty pickle file for {pfam_id} ({e}). Will re-fetch data.")
            return None
        except Exception as e:
            # other potential loading errors
            print(f"  Unexpected error loading pickle for {pfam_id}: {e}. Will re-fetch data.")
            return None
    return None

#gets all uniprot accessions for a pfam ID
def fetch_uniprot_search_results(pfam_id, base_url, fields, size=500):
    query = f"xref:PFAM-{pfam_id}"
    initial_url = f"{base_url}?query={query}&format=json&fields={fields}&size={size}"
    all_results = []
    current_url = initial_url
    while current_url:
        print(f"  Fetching UniProt search page...")
        try:
            response = requests.get(current_url)
            response.raise_for_status()
            data = response.json()

            all_results.extend(data.get('results', []))
            # Find the URL for the next page of results
            next_link = None
            for link in data.get('links', []):
                if link.get('rel') == 'next':
                    next_link = link.get('url')
                    break
            current_url = next_link
            if current_url:
                time.sleep(1) # Add a small delay between requests

        except requests.exceptions.RequestException as e:
            print(f"  Error fetching page from UniProt for PFAM {pfam_id}: {e}")
            break # Stop fetching if there's a request error

    return all_results

# checks both'recommendedName' and 'submissionNames' for proteins
def extract_protein_name_from_uniprot_result(result):
    protein_name_raw = ""
    # Try to get the recommended name first
    recommended_name_parts = result.get('proteinDescription', {}).get('recommendedName', {}).get('fullName', [])
    if isinstance(recommended_name_parts, list):
        for item in recommended_name_parts:
            if isinstance(item, dict) and 'value' in item:
                protein_name_raw = item['value'].lower()
                break
    # Fallback to submission names if recommended name is not found
    if not protein_name_raw:
        submission_names = result.get('proteinDescription', {}).get('submissionNames', [])
        if isinstance(submission_names, list):
            for item in submission_names:
                if isinstance(item, dict) and 'fullName' in item:
                    full_name_parts = item.get('fullName', [])
                    if isinstance(full_name_parts, list):
                        for sub_item in full_name_parts:
                            if isinstance(sub_item, dict) and 'value' in sub_item:
                                protein_name_raw = sub_item['value'].lower()
                                break
                        if protein_name_raw:
                            break
    return protein_name_raw

#Fetches a full Swiss-Prot record using ExPASy
def get_swissprot_record_with_retry(uniprot_accession, max_retries=3):
    record = None
    for attempt in range(max_retries):
        try:
            with ExPASy.get_sprot_raw(uniprot_accession) as handle:
                record = SwissProt.read(handle)
            break # Break the loop if successful
        except Exception as e:
            print(f"    Error fetching SwissProt for {uniprot_accession} (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2) # Wait before retrying
            else:
                record = None # Set to None if all retries fail
    return record

#fetches NCBI taxanomy ID from swissprot
def extract_taxonomy_id(record):
    taxonomy_id = None
    if record and record.taxonomy_id:
        taxonomy_id = ", ".join(record.taxonomy_id)
    else:
        # Fallback to looking in cross-references if the direct attribute is empty
        for xref in record.cross_references:
            if len(xref) >= 3 and 'taxon:' in xref[2]:
                try:
                    taxon_str_parts = [part.strip() for part in xref[2].split(';') if 'taxon:' in part]
                    if taxon_str_parts:
                        taxonomy_id = taxon_str_parts[0].split(':')[-1].strip()
                        break
                except Exception:
                    # Continue if parsing a specific cross-reference fails
                    pass
    return taxonomy_id

# Processes a single UniProt accession to get detailed Swiss-Prot data.
def process_uniprot_accession(pfam_id, uniprot_accession, protein_name_from_search):
    # Skip based on keywords found in the initial search result
    if any(keyword in protein_name_from_search for keyword in METHYLASE_KEYWORDS):
        print(f"  Skipping UniProt Accession: {uniprot_accession} (identified as methylase by keyword)")
        return None

    print(f"  Fetching detailed data for UniProt Accession: {uniprot_accession}")
    record = get_swissprot_record_with_retry(uniprot_accession)

    if record:
        taxonomy_id = extract_taxonomy_id(record)
        return {
            'PFAM ID': pfam_id,
            'UniProt Accession': uniprot_accession,
            'Entry Name': record.entry_name,
            'Protein Name': record.description,
            'Organism': record.organism,
            'Taxonomy ID': taxonomy_id,
            'Sequence Length': record.sequence_length,
            'Taxonomy': record.organism_classification,
            'Superkingdom': record.organism_classification[0] if record.organism_classification else None,
        }
    else:
        print(f"    Could not retrieve full record for {uniprot_accession}. Appending a placeholder.")
        return {
            'PFAM ID': pfam_id,
            'UniProt Accession': uniprot_accession,
            'Entry Name': None, 'Protein Name': None, 'Organism': None,
            'Taxonomy ID': None, 'Sequence Length': None, 'Taxonomy': None,
            'Superkingdom': None
        }

# Main function to fetch or load data for a single PFAM ID.
def get_uniprot_data_for_pfam(pfam_id):
    print(f"\n--- Processing PFAM ID: {pfam_id} ---")

    # Check if the data for this PFAM ID is already cached.
    cached_data = load_from_pickle(pfam_id, PICKLE_DIR)
    if cached_data is not None:
        # If cached data exists, we don't need to re-fetch anything.
        return cached_data

    #If no cached data exists, fetch from UniProt.
    print(f"  No valid cache found for {pfam_id}. Fetching from UniProt.")
    all_accessions_data_for_pfam = []
    try:
        # Fetch initial search results for all accessions linked to this PFAM ID
        search_results = fetch_uniprot_search_results(pfam_id, UNIPROT_BASE_URL, "accession,id,protein_name")

        # Process each accession found in the search results
        for result in search_results:
            uniprot_accession = result['primaryAccession']
            protein_name_from_search = extract_protein_name_from_uniprot_result(result)

            # Check for skipping based on explicit PFAM ID or protein name keywords
            if str(pfam_id) in METHYLASE_PFAM_IDS_TO_SKIP:
                print(f"  Skipping all UniProt accessions for PFAM ID {pfam_id} (explicitly skipped PFAM).")
                continue

            processed_data = process_uniprot_accession(pfam_id, uniprot_accession, protein_name_from_search)
            if processed_data:
                all_accessions_data_for_pfam.append(processed_data)

    except Exception as e:
        print(f"An unexpected error occurred while fetching data for PFAM {pfam_id}: {e}")

    # Save the newly fetched data to a pickle file for future runs.
    save_to_pickle(all_accessions_data_for_pfam, pfam_id, PICKLE_DIR)
    return all_accessions_data_for_pfam


def main():
    """
    Main function to orchestrate the entire data fetching, processing, and saving pipeline.
    """
    # Make sure the Google Drive directory is accessible.
    print(f"Checking for pickle directory at: {PICKLE_DIR}")
    if not os.path.exists(PICKLE_DIR):
        print(f"Pickle directory does not exist. Creating it: {PICKLE_DIR}")
        os.makedirs(PICKLE_DIR, exist_ok=True)
        time.sleep(2)
    else:
        print("Pickle directory found.")

    try:
        # Load the list of PFAM IDs from the Excel file.
        pfam_ids = load_pfam_ids(INPUT_EXCEL_FILE, PFAM_ID_COLUMN_NAME)

        all_results_list = []
        # Iterate through each PFAM ID
        for pfam_id in pfam_ids:
            results_for_id = get_uniprot_data_for_pfam(pfam_id)
            all_results_list.extend(results_for_id)
            time.sleep(1)

        # Consolidate all collected data into a single DataFrame.
        if all_results_list:
            final_df = pd.DataFrame(all_results_list)
            desired_columns = [
                'PFAM ID', 'UniProt Accession', 'Entry Name', 'Protein Name', 'Organism',
                'Taxonomy ID', 'Sequence Length', 'Taxonomy', 'Superkingdom'
            ]
            final_df = final_df[desired_columns]

            #Save the final DataFrame to an Excel file.
            final_df.to_excel(OUTPUT_EXCEL_FILE, index=False)
            print(f"\nData successfully saved to '{OUTPUT_EXCEL_FILE}'")
        else:
            print("\nNo data was collected. The output Excel file will not be created.")

    except ValueError as ve:
        print(f"Configuration Error: {ve}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

#entrypoint
if __name__ == "__main__":
    main()

Checking for pickle directory at: /content/drive/MyDrive/Pfam Pickles
Pickle directory found.
Reading PFAM IDs from 'RNA Modification in Viruses.xlsx'...
Found 177 unique PFAM IDs to process.

--- Processing PFAM ID: PF03481 ---
  Cache found for PF03481. Attempting to load from /content/drive/MyDrive/Pfam Pickles/pfam_PF03481.pkl...

--- Processing PFAM ID: PF13649 ---
  Cache found for PF13649. Attempting to load from /content/drive/MyDrive/Pfam Pickles/pfam_PF13649.pkl...

--- Processing PFAM ID: PF05063 ---
  Cache found for PF05063. Attempting to load from /content/drive/MyDrive/Pfam Pickles/pfam_PF05063.pkl...

--- Processing PFAM ID: PF09445 ---
  Cache found for PF09445. Attempting to load from /content/drive/MyDrive/Pfam Pickles/pfam_PF09445.pkl...

--- Processing PFAM ID: PF01134 ---
  Cache found for PF01134. Attempting to load from /content/drive/MyDrive/Pfam Pickles/pfam_PF01134.pkl...

--- Processing PFAM ID: PF03919 ---
  Cache found for PF03919. Attempting to load from 

this code links the pfam id and all superkingdom counts to each original name and ID and spits out an excel sheet with all the info

In [ ]:
import pandas as pd
import os


INPUT_VIRUS_FILE = "RNA Modification in Viruses.xlsx"
INPUT_SUMMARY_FILE = "Superkingdom_Counts_Summary.xlsx"
OUTPUT_COMBINED_FILE = "full names and superkingdom counts.xlsx"

def combine_data_and_save(virus_file, summary_file, output_file):
    """
    Combines data from the virus modification Excel file and the
    superkingdom counts summary, then saves the combined data to a new Excel file.
    """
    print(f"Attempting to load data from '{virus_file}' and '{summary_file}'...")


    try:
        df_virus = pd.read_excel(virus_file)
        df_virus_selected = df_virus[['ID', 'Traditional Name', 'Pfam ID', 'Full name']].copy()
        df_virus_selected['Pfam ID'] = df_virus_selected['Pfam ID'].astype(str).str.strip()
        print(f"Successfully loaded '{virus_file}'.")
    except FileNotFoundError:
        print(f"Error: Input file '{virus_file}' not found. Please ensure it is uploaded directly to your Colab session.")
        return
    except KeyError as e:
        print(f"Error: Missing expected column in '{virus_file}': {e}. Please check column names.")
        return
    except Exception as e:
        print(f"An unexpected error occurred while loading '{virus_file}': {e}")
        return


    try:
        df_summary = pd.read_excel(summary_file)

        df_summary['Pfam ID'] = df_summary['Pfam ID'].astype(str).str.strip()
        print(f"Successfully loaded '{summary_file}'.")
    except FileNotFoundError:
        print(f"Error: Input file '{summary_file}' not found. Please ensure it is uploaded directly to your Colab session.")
        return
    except Exception as e:
        print(f"An unexpected error occurred while loading '{summary_file}': {e}")
        return


    print("Merging dataframes based on 'Pfam ID'...")
    combined_df = pd.merge(df_virus_selected, df_summary, on='Pfam ID', how='left')

    combined_df['Superkingdom'] = combined_df['Superkingdom'].fillna('N/A')
    combined_df['Count'] = combined_df['Count'].fillna(0).astype(int)


    final_column_order = [
        'ID', 'Traditional Name', 'Pfam ID', 'Full name',
        'Superkingdom', 'Count'
    ]
    combined_df = combined_df[final_column_order]

    try:
        combined_df.to_excel(output_file, index=False)
        print(f"\nSuccessfully created and saved the combined data to '{output_file}'.")
    except Exception as e:
        print(f"Error saving the combined data to '{output_file}': {e}")


if __name__ == "__main__":

    combine_data_and_save(INPUT_VIRUS_FILE, INPUT_SUMMARY_FILE, OUTPUT_COMBINED_FILE)

Attempting to load data from 'RNA Modification in Viruses.xlsx' and 'Superkingdom_Counts_Summary.xlsx'...
Successfully loaded 'RNA Modification in Viruses.xlsx'.
Successfully loaded 'Superkingdom_Counts_Summary.xlsx'.
Merging dataframes based on 'Pfam ID'...

Successfully created and saved the combined data to 'full names and superkingdom counts.xlsx'.


this will code for pickles that store data about the rna modifications including substrates and products of each then output an excel file with everything. This excel file puts both substrates and products in one column so I had to go in by hand and correct this.

In [ ]:
import pandas as pd
import os
import pickle
import requests
import time
import re
from Bio import ExPASy
from Bio import SwissProt

INPUT_EXCEL_FILE = "RNA Modification in Viruses.xlsx"
ENZYME_NAME_COLUMN = "UniProt"
OUTPUT_EXCEL_FILE = "Enzyme_Catalytic_Activity.xlsx"

# Google Drive path and local directory for saving and loading pickle files
GOOGLE_DRIVE_MOUNT_PATH = "/content/drive/MyDrive"
PICKLE_DIR = os.path.join(GOOGLE_DRIVE_MOUNT_PATH, "Catalytic Pickles")

# Base URL for UniProt search
UNIPROT_BASE_URL = "https://rest.uniprot.org/uniprotkb/search"

#test functions to make sure the excel file can be made
def create_mock_excel(filename=INPUT_EXCEL_FILE):
    print(f"Creating mock Excel file: '{filename}'...")
    data = {
        ENZYME_NAME_COLUMN: [
            "tRNA (uracil-5-)-methyltransferase",
            "Ribosomal RNA (guanine-N1)-methyltransferase",
            "RNA (adenosine-N6)-methyltransferase",
            "DNA (adenine-N6)-methyltransferase",
            "nsp13",  # An example that might have a different search result format
            "Trm82"   # An example that caused a previous error, now handled
        ]
    }
    df = pd.DataFrame(data)
    df.to_excel(filename, index=False)
    print("Mock Excel file created successfully.")

#check if there is already a pickle for the enzyme
def load_enzyme_names(excel_file, column_name):

    print(f"Reading enzyme names from '{excel_file}'...")
    try:
        df_input = pd.read_excel(excel_file)
    except FileNotFoundError:
        print(f"Error: Input file '{excel_file}' not found. Please ensure it's in the correct directory.")
        return []
    except pd.errors.EmptyDataError:
        print(f"Error: '{excel_file}' is empty.")
        return []

    if column_name not in df_input.columns:
        raise ValueError(f"Column '{column_name}' not found in '{excel_file}'. "
                         "Please check the column name in your Excel file.")

    enzyme_names = df_input[column_name].dropna().unique().tolist()
    print(f"Found {len(enzyme_names)} unique enzymes to process.")
    return [str(name).strip() for name in enzyme_names]

#saving new pickle to folder
def save_to_pickle(data, enzyme_name, directory):

    try:
        os.makedirs(directory, exist_ok=True)
        filename = "".join(c for c in enzyme_name if c.isalnum() or c in (' ', '_', '-')).rstrip()
        filename = filename.replace(' ', '_')
        pickle_file_path = os.path.join(directory, f"{filename}.pkl")

        with open(pickle_file_path, 'wb') as f:
            pickle.dump(data, f)
        print(f"  Data for '{enzyme_name}' saved to {pickle_file_path}")
    except Exception as e:
        print(f"  Error saving pickle file for '{enzyme_name}': {e}")

#if there is already a pickle it will load it
def load_from_pickle(enzyme_name, directory):

    filename = "".join(c for c in enzyme_name if c.isalnum() or c in (' ', '_', '-')).rstrip()
    filename = filename.replace(' ', '_')
    pickle_file_path = os.path.join(directory, f"{filename}.pkl")

    if os.path.exists(pickle_file_path) and os.path.getsize(pickle_file_path) > 0:
        print(f"  Cache found for '{enzyme_name}'. Attempting to load...")
        try:
            with open(pickle_file_path, 'rb') as f:
                return pickle.load(f)
        except (EOFError, pickle.UnpicklingError) as e:
            print(f"  Error loading from corrupted/empty pickle file for '{enzyme_name}' ({e}). Will re-fetch data.")
            return None
        except Exception as e:
            print(f"  Unexpected error loading pickle for '{enzyme_name}': {e}. Will re-fetch data.")
            return None
    return None

#gets swiss prot record of each enzyme and also retries if there is connection issue
def get_swissprot_record_with_retry(uniprot_accession, max_retries=3):

    record = None
    for attempt in range(max_retries):
        try:
            with ExPASy.get_sprot_raw(uniprot_accession) as handle:
                record = SwissProt.read(handle)
            break
        except Exception as e:
            print(f"    Error fetching SwissProt for {uniprot_accession} (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                record = None
    return record

#what parses and saves catalytic info *THIS IS WHAT I THINK NEEDS ADJUSTMENT* because it will take only the first substrate and product when there might be multiple
def extract_catalytic_info_from_record(record):

    substrates = "Not found"
    products = "Not found"

    for comment in record.comments:
        if comment.startswith("CATALYTIC ACTIVITY:"):
            catalytic_activity = comment.split("CATALYTIC ACTIVITY:")[1].strip()


            if '=' in catalytic_activity:
                parts = catalytic_activity.split('=', 1)


                substrates_str = parts[0].strip()

                substrates_str = re.sub(r'^Reaction:\s*', '', substrates_str, flags=re.IGNORECASE).strip()


                products_str = parts[1].strip()

                products_str = re.split(r'[;\[]', products_str, 1)[0].strip()

                substrates = [s.strip() for s in substrates_str.split('+') if s.strip()]
                products = [p.strip() for p in products_str.split('+') if p.strip()]

            else:

                substrate_match = re.search(r"Substrate:\s*(.+?)(?=\s*Product:|\.|\s*\[|$)", catalytic_activity, re.DOTALL)
                product_match = re.search(r"Product:\s*(.+?)(?=\.|\s*\[|$)", catalytic_activity, re.DOTALL)

                if substrate_match:
                    substrates_str = re.sub(r'\s+', ' ', substrate_match.group(1)).strip()
                    substrates = [s.strip() for s in substrates_str.split(',')]

                if product_match:
                    products_str = re.sub(r'\s+', ' ', product_match.group(1)).strip()
                    products = [p.strip() for p in products_str.split(',')]

            break

    return substrates, products

#This is the main function for a single enzyme. It first tries to load from the cache.
#If it fails it queries the UniProt search API for a unique UniProt ID, then uses that ID to fetch the full Swiss-Prot record, extracts the catalytic info, and finally saves the result to a pickle file.
def get_uniprot_data_for_enzyme(enzyme_name):
    """
    Main function to fetch or load data for a single enzyme.

    Args:
        enzyme_name (str): The name of the enzyme to process.

    Returns:
        dict: A dictionary containing the enzyme's catalytic info, or None if not found.
    """
    print(f"\n--- Processing Enzyme: '{enzyme_name}' ---")

    cached_data = load_from_pickle(enzyme_name, PICKLE_DIR)
    if cached_data is not None:
        return cached_data

    print(f"  No valid cache found for '{enzyme_name}'. Fetching from UniProt.")
    try:

        query_url = "https://rest.uniprot.org/uniprotkb/search"
        params = {
            "query": f'"{enzyme_name}"',
            "format": "json",
            "size": 1
        }

        response = requests.get(query_url, params=params)
        response.raise_for_status()
        data = response.json()

        record_id = None
        if data.get('results') and len(data['results']) > 0:
            record_id = data['results'][0]['primaryAccession']

        if not record_id:
            print(f"  - No UniProt ID found for '{enzyme_name}'. Skipping.")
            return None

        print(f"  - Found UniProt ID: {record_id}")


        record = get_swissprot_record_with_retry(record_id)

        if record:
            substrates, products = extract_catalytic_info_from_record(record)

            enzyme_data = {
                "Enzyme": enzyme_name,
                "Substrates": substrates,
                "Products": products
            }


            save_to_pickle(enzyme_data, enzyme_name, PICKLE_DIR)
            return enzyme_data
        else:
            print(f"  - Could not retrieve full record for '{enzyme_name}'. Appending a placeholder.")
            return {
                "Enzyme": enzyme_name,
                "Substrates": "Not found",
                "Products": "Not found"
            }

    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching data for '{enzyme_name}' from UniProt: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while processing '{enzyme_name}': {e}")
        return None

# creates excel file with all collected data corresponding to that enzyme.
def create_output_excel(all_data, output_filename=OUTPUT_EXCEL_FILE):
    if not all_data:
        print("No data collected to write to Excel. Exiting.")
        return

    print(f"\nCreating output Excel file: '{output_filename}'...")
    df = pd.DataFrame(all_data)


    df['Substrates'] = df['Substrates'].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
    df['Products'] = df['Products'].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)

    df.to_excel(output_filename, index=False)
    print("Output Excel file created successfully.")

#This function ties all the other functions together. It checks for the Google Drive directory, creates a mock Excel file if needed, loads the enzyme names, loops through each enzyme to fetch its data using get_uniprot_data_for_enzyme, and finally calls create_output_excel to save the results.
def main():
    print(f"Checking for Google Drive directory at: {PICKLE_DIR}")
    if not os.path.exists(GOOGLE_DRIVE_MOUNT_PATH):
        print(f"Error: Google Drive path '{GOOGLE_DRIVE_MOUNT_PATH}' does not exist.")
        print("Please ensure your Google Drive is mounted in Google Colab before running the script.")
        return
    else:
        print("Google Drive directory found.")
    if not os.path.exists(INPUT_EXCEL_FILE):
        create_mock_excel(INPUT_EXCEL_FILE)

    try:
        enzyme_names = load_enzyme_names(INPUT_EXCEL_FILE, ENZYME_NAME_COLUMN)
    except ValueError as ve:
        print(f"Configuration Error: {ve}")
        return

    all_results_list = []
    for i, name in enumerate(enzyme_names):
        results_for_name = get_uniprot_data_for_enzyme(name)

        if i == 0 and results_for_name:
            print("\n--- Substrates and Products for the First Enzyme ---")
            print(f"Enzyme: {results_for_name['Enzyme']}")
            print(f"Substrates: {results_for_name['Substrates']}")
            print(f"Products: {results_for_name['Products']}")
            print("---------------------------------------------------\n")

        if results_for_name:
            all_results_list.append(results_for_name)
        time.sleep(1)

    create_output_excel(all_results_list, OUTPUT_EXCEL_FILE)

    print("\nProcess completed successfully!")

if __name__ == "__main__":
    main()

Checking for Google Drive directory at: /content/drive/MyDrive/Catalytic Pickles
Google Drive directory found.
Reading enzyme names from 'RNA Modification in Viruses.xlsx'...
Found 475 unique enzymes to process.

--- Processing Enzyme: 'P32579' ---
  No valid cache found for 'P32579'. Fetching from UniProt.
  - Found UniProt ID: P32579
  Data for 'P32579' saved to /content/drive/MyDrive/Catalytic Pickles/P32579.pkl

--- Substrates and Products for the First Enzyme ---
Enzyme: P32579
Substrates: ['Reaction']
Products: ['L-threonine', 'hydrogencarbonate', 'ATP = L-   threonylcarbamoyladenylate', 'diphosphate', 'H2O']
---------------------------------------------------


--- Processing Enzyme: 'P36999' ---
  No valid cache found for 'P36999'. Fetching from UniProt.
  - Found UniProt ID: P36999
  Data for 'P36999' saved to /content/drive/MyDrive/Catalytic Pickles/P36999.pkl

--- Processing Enzyme: 'P41833' ---
  No valid cache found for 'P41833'. Fetching from UniProt.
  - Found UniProt ID

Counting species per domains for each pfam

In [ ]:
import os
import glob
import pandas as pd
import re
from google.colab import drive

drive.mount('/content/drive')
folder_path = "/content/drive/MyDrive/PFAM TSV"
output_file = "/content/drive/MyDrive/pfam_domain_counts.xlsx"


target_domains = ["Eukaryota", "Bacteria", "Viruses", "Archaea"]

summary_data = []

print(f"looking for TSV files in: {folder_path}")
tsv_files = glob.glob(os.path.join(folder_path, "*.tsv"))

if not tsv_files:
    print(f"no TSV files found in {folder_path}!")
else:
    print(f"Found {len(tsv_files)} files. Starting processing...")

    for file_path in tsv_files:
        filename = os.path.basename(file_path)

        # Extract PFAM ID using regex
        match = re.search(r"(PF\d+)", filename)
        pfam_id = match.group(1) if match else filename

        try:
            # Load only the necessary column
            df = pd.read_csv(file_path, sep='\t', usecols=['Taxonomic lineage'])

            # Normalize to lowercase
            lineages = df['Taxonomic lineage'].dropna().astype(str).str.lower()

            # Hash Map for counts (O(1) access)
            counts = {domain: 0 for domain in target_domains}

            # Linear Scan (O(N))
            for lineage in lineages:
                for domain in target_domains:
                    if domain.lower() in lineage:
                        counts[domain] += 1

            # Store results
            row = {'Pfam ID': pfam_id}
            row.update(counts)
            summary_data.append(row)

            print(f"Processed {pfam_id}")

        except ValueError:
            print(f"Skipping {filename}: 'Taxonomic lineage' column not found.")
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    if summary_data:
        result_df = pd.DataFrame(summary_data)

        # Reorder columns
        cols = ['Pfam ID'] + target_domains
        result_df = result_df[cols]

        # Save to Drive
        result_df.to_excel(output_file, index=False)
        print("-" * 30)
        print(f"Success! Processed {len(summary_data)} files.")
        print(f"File saved to your Drive at: {output_file}")
    else:
        print("No data processed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
looking for TSV files in: /content/drive/MyDrive/PFAM TSV
Found 38 files. Starting processing...
Processed PF08704
Processed PF00145
Processed PF00258
Processed PF00749
Processed PF01134
Processed PF01163
Processed PF01171
Processed PF01207
Processed PF01266
Processed PF01416
Processed PF01596
Processed PF01702
Processed PF01715
Processed PF01728
Processed PF01746
Processed PF01926
Processed PF02475
Processed PF02547
Processed PF02926
Processed PF03054
Processed PF03481
Processed PF03966
Processed PF04013
Processed PF04055
Processed PF04179
Processed PF04816
Processed PF06175
Processed PF08489
Processed PF12934
Processed PF13418
Processed PF13484
Processed PF13621
Processed PF13649
Processed PF14437
Processed PF15387
Processed PF17884
Processed PF18389
Processed PF00586
------------------------------
Success! Processed 38 files.
File saved to your Drive at: /

this code and the next few cells for the other domains, uses the pathway code sheet to link enzyme pfam organisms to each pathway and defines if it is complete or not. If an organism conatins an enzyme it will record the pfam and uniprot ID of that enzyme.

for virus

In [ ]:
import pandas as pd
import os
import glob
import time
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def call_with_retry(func, *args, retries=5, delay=2, **kwargs):
    """Handles API 500 errors using exponential backoff to ensure pipeline stability."""
    for i in range(retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if i == retries - 1:
                raise e
            print(f"  [RETRY] API encounter. Waiting {delay}s...")
            time.sleep(delay)
            delay *= 2

def generate_viral_pathway_analysis():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')
    # Dedicated folder for Viral results
    output_folder = os.path.join(base_path, 'Viral_Pathway_Analysis_Outputs')

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created output directory: {output_folder}")

    # 2. Load and Normalize Sheet Data
    try:
        print("Connecting to Google Sheets...")
        spreadsheet = call_with_retry(gc.open_by_key, spreadsheet_id)

        enzyme_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("enzyme info").get_all_records))
        pathway_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("pathway data").get_all_records))
        gene_mapping_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("Gene pfam").get_all_records))

        for df in [enzyme_df, pathway_df, gene_mapping_df]:
            df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"CRITICAL: API Failure: {e}")
        return

    # 3. Build Mappings (Hash Tables)
    gene_to_metadata = {
        str(row['Gene']).strip().upper(): str(row['pfam ID']).strip()
        for _, row in gene_mapping_df.iterrows()
    }

    path_to_steps = {}
    for _, row in pathway_df.iterrows():
        p_name = str(row['Pathway Name']).strip()
        steps = []
        gene_cols = [c for c in row.index if 'Gene' in c]
        for col in gene_cols:
            if pd.notna(row[col]) and str(row[col]).strip() != "":
                # Handle logical OR for steps with multiple potential genes
                step_genes = [g.strip() for g in str(row[col]).replace(';', ',').split(',')]
                steps.append(step_genes)
        path_to_steps[p_name] = steps

    # 4. Index Files and Setup Cache
    all_tsvs = glob.glob(os.path.join(pfam_folder, "*.tsv"))
    # Extracts Pfam ID from 'uniprotkb_PFXXXXX_date.tsv'
    pfam_file_lookup = {os.path.basename(f).split('_')[1].strip(): f for f in all_tsvs if len(os.path.basename(f).split('_')) > 1}

    pfam_data_cache = {}

    def get_pfam_data(pfam_id):
        """Returns a dict mapping lower-case organism names to their Uniprot Entry IDs."""
        if pfam_id in pfam_data_cache:
            return pfam_data_cache[pfam_id]

        path = pfam_file_lookup.get(pfam_id)
        org_map = {}
        if path:
            try:
                # DSA Efficiency: Load only the required columns to save memory
                df = pd.read_csv(path, sep='\t', usecols=['Organism', 'Entry'], low_memory=False)
                for _, row in df.iterrows():
                    org_name = str(row['Organism']).strip().lower()
                    entry_id = str(row['Entry']).strip()
                    org_map[org_name] = entry_id
            except Exception:
                pass

        pfam_data_cache[pfam_id] = org_map
        return org_map

    # 5. Iterative Processing (Full Sheet)
    print(f"Processing {len(enzyme_df)} enzymes for Viral analysis...")

    for idx, target_row in enzyme_df.iterrows():
        current_pfam = str(target_row['pfam ID']).strip()
        # Handle enzymes involved in multiple pathways
        pathway_list = [p.strip() for p in str(target_row['Pathway Name']).replace(';', ',').split(',') if p.strip()]

        out_file = os.path.join(output_folder, f'Viral_Matrix_{current_pfam}.xlsx')
        primary_path = pfam_file_lookup.get(current_pfam)

        if not primary_path:
            pd.DataFrame({"Status": ["File not found"]}).to_excel(out_file, index=False)
            continue

        # Load first enzyme TSV to find target Viruses
        primary_df = pd.read_csv(primary_path, sep='\t', low_memory=False)
        primary_df.columns = primary_df.columns.str.strip()

        lineage_col = 'Taxonomic lineage' if 'Taxonomic lineage' in primary_df.columns else 'Organism'
        # Search Taxonomic lineage for "Viruses"
        v_mask = primary_df[lineage_col].str.contains('Viruses', case=False, na=False)
        unique_viruses = sorted(primary_df[v_mask]['Organism'].unique())

        if not unique_viruses:
            pd.DataFrame({"Status": ["No Viruses Found"]}).to_excel(out_file, index=False)
            continue

        results = []
        for v_name in unique_viruses:
            v_lower = v_name.strip().lower()
            row_data = {"Organism": v_name}

            for p_name in pathway_list:
                required_steps = path_to_steps.get(p_name, [])
                for i, step_genes in enumerate(required_steps, 1):
                    col_header = f"{p_name} Step {i}"
                    found_matches = []

                    for gene in step_genes:
                        pfam_id = gene_to_metadata.get(gene.upper())
                        if pfam_id:
                            # Search the data cache for this specific Pfam
                            org_lookup = get_pfam_data(pfam_id)
                            if v_lower in org_lookup:
                                entry_id = org_lookup[v_lower]
                                # REQUIRED FORMAT: Pfam ID, Entry ID
                                found_matches.append(f"{pfam_id}, {entry_id}")

                    # Join multiple gene hits in the same step with a semicolon
                    row_data[col_header] = "; ".join(list(set(found_matches))) if found_matches else "missing"

            results.append(row_data)

        if results:
            pd.DataFrame(results).to_excel(out_file, index=False)

        # Exponential backoff/sleep to protect API
        time.sleep(0.5)

    print(f"\nCompleted. Results available in: {output_folder}")

# Execute the Viral analysis
generate_viral_pathway_analysis()

Mounted at /content/drive
Created output directory: /content/drive/My Drive/Lauren Paprocki/Viral_Pathway_Analysis_Outputs
Connecting to Google Sheets...
Processing 110 enzymes for Viral analysis...

Completed. Results available in: /content/drive/My Drive/Lauren Paprocki/Viral_Pathway_Analysis_Outputs


for archaea

In [ ]:
import pandas as pd
import os
import glob
import time
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def call_with_retry(func, *args, retries=5, delay=2, **kwargs):
    """Handles API 500 errors using exponential backoff to ensure pipeline stability."""
    for i in range(retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if i == retries - 1:
                raise e
            print(f"  [RETRY] API encounter. Waiting {delay}s...")
            time.sleep(delay)
            delay *= 2

def generate_archaea_pathway_analysis():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')
    # Dedicated folder for Archaea results
    output_folder = os.path.join(base_path, 'Archaea_Pathway_Analysis_Outputs')

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created output directory: {output_folder}")

    # 2. Load and Normalize Sheet Data
    try:
        print("Connecting to Google Sheets...")
        spreadsheet = call_with_retry(gc.open_by_key, spreadsheet_id)

        enzyme_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("enzyme info").get_all_records))
        pathway_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("pathway data").get_all_records))
        gene_mapping_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("Gene pfam").get_all_records))

        for df in [enzyme_df, pathway_df, gene_mapping_df]:
            df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"CRITICAL: API Failure: {e}")
        return

    # 3. Build Mappings (Hash Tables)
    gene_to_metadata = {
        str(row['Gene']).strip().upper(): str(row['pfam ID']).strip()
        for _, row in gene_mapping_df.iterrows()
    }

    path_to_steps = {}
    for _, row in pathway_df.iterrows():
        p_name = str(row['Pathway Name']).strip()
        steps = []
        gene_cols = [c for c in row.index if 'Gene' in c]
        for col in gene_cols:
            if pd.notna(row[col]) and str(row[col]).strip() != "":
                # Handle logical OR for steps with multiple potential genes
                step_genes = [g.strip() for g in str(row[col]).replace(';', ',').split(',')]
                steps.append(step_genes)
        path_to_steps[p_name] = steps

    # 4. Index Files and Setup Cache
    all_tsvs = glob.glob(os.path.join(pfam_folder, "*.tsv"))
    # Extracts Pfam ID from 'uniprotkb_PFXXXXX_date.tsv'
    pfam_file_lookup = {os.path.basename(f).split('_')[1].strip(): f for f in all_tsvs if len(os.path.basename(f).split('_')) > 1}

    pfam_data_cache = {}

    def get_pfam_data(pfam_id):
        """Returns a dict mapping lower-case organism names to their Uniprot Entry IDs."""
        if pfam_id in pfam_data_cache:
            return pfam_data_cache[pfam_id]

        path = pfam_file_lookup.get(pfam_id)
        org_map = {}
        if path:
            try:
                # Efficiency: Load only the required columns to save memory
                df = pd.read_csv(path, sep='\t', usecols=['Organism', 'Entry'], low_memory=False)
                for _, row in df.iterrows():
                    org_name = str(row['Organism']).strip().lower()
                    entry_id = str(row['Entry']).strip()
                    org_map[org_name] = entry_id
            except Exception:
                pass

        pfam_data_cache[pfam_id] = org_map
        return org_map

    # 5. Iterative Processing (Full Sheet)
    print(f"Processing {len(enzyme_df)} enzymes for Archaea analysis...")

    for idx, target_row in enzyme_df.iterrows():
        current_pfam = str(target_row['pfam ID']).strip()
        # Handle enzymes involved in multiple pathways
        pathway_list = [p.strip() for p in str(target_row['Pathway Name']).replace(';', ',').split(',') if p.strip()]

        out_file = os.path.join(output_folder, f'Archaea_Matrix_{current_pfam}.xlsx')
        primary_path = pfam_file_lookup.get(current_pfam)

        if not primary_path:
            pd.DataFrame({"Status": ["File not found"]}).to_excel(out_file, index=False)
            continue

        # Load first enzyme TSV to find target Archaea
        primary_df = pd.read_csv(primary_path, sep='\t', low_memory=False)
        primary_df.columns = primary_df.columns.str.strip()

        lineage_col = 'Taxonomic lineage' if 'Taxonomic lineage' in primary_df.columns else 'Organism'
        # Search Taxonomic lineage for "Archaea"
        a_mask = primary_df[lineage_col].str.contains('Archaea', case=False, na=False)
        unique_archaea = sorted(primary_df[a_mask]['Organism'].unique())

        if not unique_archaea:
            pd.DataFrame({"Status": ["No Archaea Found"]}).to_excel(out_file, index=False)
            continue

        results = []
        for a_name in unique_archaea:
            a_lower = a_name.strip().lower()
            row_data = {"Organism": a_name}

            for p_name in pathway_list:
                required_steps = path_to_steps.get(p_name, [])
                for i, step_genes in enumerate(required_steps, 1):
                    col_header = f"{p_name} Step {i}"
                    found_matches = []

                    for gene in step_genes:
                        pfam_id = gene_to_metadata.get(gene.upper())
                        if pfam_id:
                            # Search the data cache for this specific Pfam
                            org_lookup = get_pfam_data(pfam_id)
                            if a_lower in org_lookup:
                                entry_id = org_lookup[a_lower]
                                # REQUIRED FORMAT: Pfam ID, Entry ID
                                found_matches.append(f"{pfam_id}, {entry_id}")

                    # Join multiple gene hits in the same step with a semicolon
                    row_data[col_header] = "; ".join(list(set(found_matches))) if found_matches else "missing"

            results.append(row_data)

        if results:
            pd.DataFrame(results).to_excel(out_file, index=False)

        # Exponential backoff/sleep to protect API
        time.sleep(0.5)

    print(f"\nCompleted. Results available in: {output_folder}")

# Execute the Archaea analysis
generate_archaea_pathway_analysis()

Mounted at /content/drive
Created output directory: /content/drive/My Drive/Lauren Paprocki/Archaea_Pathway_Analysis_Outputs
Connecting to Google Sheets...
Processing 110 enzymes for Archaea analysis...

Completed. Results available in: /content/drive/My Drive/Lauren Paprocki/Archaea_Pathway_Analysis_Outputs


for Eukaryotes

In [ ]:
import pandas as pd
import os
import glob
import time
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def call_with_retry(func, *args, retries=5, delay=2, **kwargs):
    """Handles API 500 errors using exponential backoff to ensure pipeline stability."""
    for i in range(retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if i == retries - 1:
                raise e
            print(f"  [RETRY] API encounter. Waiting {delay}s...")
            time.sleep(delay)
            delay *= 2

def generate_eukaryotic_pathway_analysis():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')
    # Dedicated folder for Eukaryotic results
    output_folder = os.path.join(base_path, 'Eukaryotic_Pathway_Analysis_Outputs')

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created output directory: {output_folder}")

    # 2. Load and Normalize Sheet Data
    try:
        print("Connecting to Google Sheets...")
        spreadsheet = call_with_retry(gc.open_by_key, spreadsheet_id)

        enzyme_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("enzyme info").get_all_records))
        pathway_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("pathway data").get_all_records))
        gene_mapping_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("Gene pfam").get_all_records))

        for df in [enzyme_df, pathway_df, gene_mapping_df]:
            df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"CRITICAL: API Failure: {e}")
        return

    # 3. Build Mappings (Hash Tables)
    gene_to_metadata = {
        str(row['Gene']).strip().upper(): str(row['pfam ID']).strip()
        for _, row in gene_mapping_df.iterrows()
    }

    path_to_steps = {}
    for _, row in pathway_df.iterrows():
        p_name = str(row['Pathway Name']).strip()
        steps = []
        gene_cols = [c for c in row.index if 'Gene' in c]
        for col in gene_cols:
            if pd.notna(row[col]) and str(row[col]).strip() != "":
                # Handle logical OR for steps with multiple potential genes
                step_genes = [g.strip() for g in str(row[col]).replace(';', ',').split(',')]
                steps.append(step_genes)
        path_to_steps[p_name] = steps

    # 4. Index Files and Setup Cache
    all_tsvs = glob.glob(os.path.join(pfam_folder, "*.tsv"))
    # Extracts Pfam ID from 'uniprotkb_PFXXXXX_date.tsv'
    pfam_file_lookup = {os.path.basename(f).split('_')[1].strip(): f for f in all_tsvs if len(os.path.basename(f).split('_')) > 1}

    pfam_data_cache = {}

    def get_pfam_data(pfam_id):
        """Returns a dict mapping lower-case organism names to their Uniprot Entry IDs."""
        if pfam_id in pfam_data_cache:
            return pfam_data_cache[pfam_id]

        path = pfam_file_lookup.get(pfam_id)
        org_map = {}
        if path:
            try:
                # Efficiency: Load only the required columns to save memory (crucial for large Eukaryotic TSVs)
                df = pd.read_csv(path, sep='\t', usecols=['Organism', 'Entry'], low_memory=False)
                for _, row in df.iterrows():
                    org_name = str(row['Organism']).strip().lower()
                    entry_id = str(row['Entry']).strip()
                    org_map[org_name] = entry_id
            except Exception:
                pass

        pfam_data_cache[pfam_id] = org_map
        return org_map

    # 5. Iterative Processing (Full Sheet)
    print(f"Processing {len(enzyme_df)} enzymes for Eukaryotic analysis...")

    for idx, target_row in enzyme_df.iterrows():
        current_pfam = str(target_row['pfam ID']).strip()
        # Handle enzymes involved in multiple pathways
        pathway_list = [p.strip() for p in str(target_row['Pathway Name']).replace(';', ',').split(',') if p.strip()]

        out_file = os.path.join(output_folder, f'Eukaryotic_Matrix_{current_pfam}.xlsx')
        primary_path = pfam_file_lookup.get(current_pfam)

        if not primary_path:
            pd.DataFrame({"Status": ["File not found"]}).to_excel(out_file, index=False)
            continue

        # Load first enzyme TSV to find target Eukaryotes
        primary_df = pd.read_csv(primary_path, sep='\t', low_memory=False)
        primary_df.columns = primary_df.columns.str.strip()

        lineage_col = 'Taxonomic lineage' if 'Taxonomic lineage' in primary_df.columns else 'Organism'
        # Search Taxonomic lineage for "Eukaryota"
        e_mask = primary_df[lineage_col].str.contains('Eukaryota', case=False, na=False)
        unique_eukaryotes = sorted(primary_df[e_mask]['Organism'].unique())

        if not unique_eukaryotes:
            pd.DataFrame({"Status": ["No Eukaryotes Found"]}).to_excel(out_file, index=False)
            continue

        results = []
        for e_name in unique_eukaryotes:
            e_lower = e_name.strip().lower()
            row_data = {"Organism": e_name}

            for p_name in pathway_list:
                required_steps = path_to_steps.get(p_name, [])
                for i, step_genes in enumerate(required_steps, 1):
                    col_header = f"{p_name} Step {i}"
                    found_matches = []

                    for gene in step_genes:
                        pfam_id = gene_to_metadata.get(gene.upper())
                        if pfam_id:
                            # Search the data cache for this specific Pfam
                            org_lookup = get_pfam_data(pfam_id)
                            if e_lower in org_lookup:
                                entry_id = org_lookup[e_lower]
                                # REQUIRED FORMAT: Pfam ID, Entry ID
                                found_matches.append(f"{pfam_id}, {entry_id}")

                    # Join multiple gene hits in the same step with a semicolon
                    row_data[col_header] = "; ".join(list(set(found_matches))) if found_matches else "missing"

            results.append(row_data)

        if results:
            pd.DataFrame(results).to_excel(out_file, index=False)

        # Protect API rate limits
        time.sleep(0.5)

    print(f"\nCompleted. Results available in: {output_folder}")

# Execute the Eukaryotic analysis
generate_eukaryotic_pathway_analysis()

Mounted at /content/drive
Created output directory: /content/drive/My Drive/Lauren Paprocki/Eukaryotic_Pathway_Analysis_Outputs
Connecting to Google Sheets...
Processing 110 enzymes for Eukaryotic analysis...

Completed. Results available in: /content/drive/My Drive/Lauren Paprocki/Eukaryotic_Pathway_Analysis_Outputs


code for bacteria

In [ ]:
import pandas as pd
import os
import glob
import time
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def call_with_retry(func, *args, retries=5, delay=2, **kwargs):
    """Handles API 500 errors using exponential backoff to ensure pipeline stability."""
    for i in range(retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if i == retries - 1:
                raise e
            print(f"  [RETRY] API encounter. Waiting {delay}s...")
            time.sleep(delay)
            delay *= 2

def generate_bacterial_pathway_analysis():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')
    # Dedicated folder for Bacterial results
    output_folder = os.path.join(base_path, 'Bacterial_Pathway_Analysis_Outputs')

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created output directory: {output_folder}")

    # 2. Load and Normalize Sheet Data
    try:
        print("Connecting to Google Sheets...")
        spreadsheet = call_with_retry(gc.open_by_key, spreadsheet_id)

        enzyme_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("enzyme info").get_all_records))
        pathway_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("pathway data").get_all_records))
        gene_mapping_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("Gene pfam").get_all_records))

        for df in [enzyme_df, pathway_df, gene_mapping_df]:
            df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"CRITICAL: API Failure: {e}")
        return

    # 3. Build Mappings (Hash Tables)
    gene_to_metadata = {
        str(row['Gene']).strip().upper(): str(row['pfam ID']).strip()
        for _, row in gene_mapping_df.iterrows()
    }

    path_to_steps = {}
    for _, row in pathway_df.iterrows():
        p_name = str(row['Pathway Name']).strip()
        steps = []
        gene_cols = [c for c in row.index if 'Gene' in c]
        for col in gene_cols:
            if pd.notna(row[col]) and str(row[col]).strip() != "":
                # Handle logical OR for steps with multiple potential genes
                step_genes = [g.strip() for g in str(row[col]).replace(';', ',').split(',')]
                steps.append(step_genes)
        path_to_steps[p_name] = steps

    # 4. Index Files and Setup Cache
    all_tsvs = glob.glob(os.path.join(pfam_folder, "*.tsv"))
    # Extracts Pfam ID from 'uniprotkb_PFXXXXX_date.tsv'
    pfam_file_lookup = {os.path.basename(f).split('_')[1].strip(): f for f in all_tsvs if len(os.path.basename(f).split('_')) > 1}

    pfam_data_cache = {}

    def get_pfam_data(pfam_id):
        """Returns a dict mapping lower-case organism names to their Uniprot Entry IDs."""
        if pfam_id in pfam_data_cache:
            return pfam_data_cache[pfam_id]

        path = pfam_file_lookup.get(pfam_id)
        org_map = {}
        if path:
            try:
                # Efficiency: Load only the required columns to save memory
                df = pd.read_csv(path, sep='\t', usecols=['Organism', 'Entry'], low_memory=False)
                for _, row in df.iterrows():
                    org_name = str(row['Organism']).strip().lower()
                    entry_id = str(row['Entry']).strip()
                    org_map[org_name] = entry_id
            except Exception:
                pass

        pfam_data_cache[pfam_id] = org_map
        return org_map

    # 5. Iterative Processing (Full Sheet)
    print(f"Processing {len(enzyme_df)} enzymes for Bacterial analysis...")

    for idx, target_row in enzyme_df.iterrows():
        current_pfam = str(target_row['pfam ID']).strip()
        # Handle enzymes involved in multiple pathways
        pathway_list = [p.strip() for p in str(target_row['Pathway Name']).replace(';', ',').split(',') if p.strip()]

        out_file = os.path.join(output_folder, f'Bacterial_Matrix_{current_pfam}.xlsx')
        primary_path = pfam_file_lookup.get(current_pfam)

        if not primary_path:
            pd.DataFrame({"Status": ["File not found"]}).to_excel(out_file, index=False)
            continue

        # Load first enzyme TSV to find target Bacteria
        primary_df = pd.read_csv(primary_path, sep='\t', low_memory=False)
        primary_df.columns = primary_df.columns.str.strip()

        lineage_col = 'Taxonomic lineage' if 'Taxonomic lineage' in primary_df.columns else 'Organism'
        # Search Taxonomic lineage for "Bacteria"
        b_mask = primary_df[lineage_col].str.contains('Bacteria', case=False, na=False)
        unique_bacteria = sorted(primary_df[b_mask]['Organism'].unique())

        if not unique_bacteria:
            pd.DataFrame({"Status": ["No Bacteria Found"]}).to_excel(out_file, index=False)
            continue

        results = []
        for b_name in unique_bacteria:
            b_lower = b_name.strip().lower()
            row_data = {"Organism": b_name}

            for p_name in pathway_list:
                required_steps = path_to_steps.get(p_name, [])
                for i, step_genes in enumerate(required_steps, 1):
                    col_header = f"{p_name} Step {i}"
                    found_matches = []

                    for gene in step_genes:
                        pfam_id = gene_to_metadata.get(gene.upper())
                        if pfam_id:
                            # Search the data cache for this specific Pfam
                            org_lookup = get_pfam_data(pfam_id)
                            if b_lower in org_lookup:
                                entry_id = org_lookup[b_lower]
                                # REQUIRED FORMAT: Pfam ID, Entry ID
                                found_matches.append(f"{pfam_id}, {entry_id}")

                    # Join multiple gene hits in the same step with a semicolon
                    row_data[col_header] = "; ".join(list(set(found_matches))) if found_matches else "missing"

            results.append(row_data)

        if results:
            pd.DataFrame(results).to_excel(out_file, index=False)

        # Protect API rate limits
        time.sleep(0.5)

    print(f"\nCompleted. Results available in: {output_folder}")

# Execute the Bacterial analysis
generate_bacterial_pathway_analysis()

Mounted at /content/drive
Created output directory: /content/drive/My Drive/Lauren Paprocki/Bacterial_Pathway_Analysis_Outputs
Connecting to Google Sheets...
Processing 110 enzymes for Bacterial analysis...

Completed. Results available in: /content/drive/My Drive/Lauren Paprocki/Bacterial_Pathway_Analysis_Outputs


Code checking if all the pfam files are in the PFAM TSV folder

In [ ]:
import pandas as pd
import os
import glob
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def scan_for_missing_pfams():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')

    # 2. Load Spreadsheet Data
    try:
        spreadsheet = gc.open_by_key(spreadsheet_id)
        # Pull Pfam IDs from both mapping and enzyme list
        gene_pfam_df = pd.DataFrame(spreadsheet.worksheet("Gene pfam").get_all_records())
        enzyme_info_df = pd.DataFrame(spreadsheet.worksheet("enzyme info").get_all_records())

        # Combine all unique Pfam IDs required for the project
        required_pfams = set()
        required_pfams.update(gene_pfam_df['pfam ID'].astype(str).str.strip().unique())
        required_pfams.update(enzyme_info_df['pfam ID'].astype(str).str.strip().unique())

        # Remove any empty strings or 'N/A'
        required_pfams = {p for p in required_pfams if p and p.lower() != 'nan' and p.lower() != 'n/a'}

    except Exception as e:
        print(f"Error accessing Google Sheet: {e}")
        return

    # 3. Index Existing Files in Drive
    # We look for the Pfam ID inside your versioned filenames (uniprotkb_PFXXXXX_date.tsv)
    print(f"Scanning folder: {pfam_folder}...")
    existing_files = glob.glob(os.path.join(pfam_folder, "*.tsv"))

    # Create a set of IDs that we actually have on disk
    found_pfam_ids = set()
    for fpath in existing_files:
        fname = os.path.basename(fpath)
        parts = fname.split('_')
        if len(parts) > 1:
            found_pfam_ids.add(parts[1].strip())

    # 4. Identify the Delta (Missing IDs)
    missing_pfams = sorted(list(required_pfams - found_pfam_ids))

    # 5. Output Results
    print("\n" + "="*30)
    print(f"SCAN COMPLETE")
    print(f"Total Unique Pfams Required: {len(required_pfams)}")
    print(f"Total Files Found in Drive: {len(found_pfam_ids)}")
    print(f"MISSING PFAMS: {len(missing_pfams)}")
    print("="*30)

    if missing_pfams:
        print("\nPlease download the TSV files for the following IDs:")
        for mp in missing_pfams:
            print(f" - {mp}")

        # Optional: Save this to a text file for easy copy-pasting
        with open(os.path.join(base_path, 'missing_pfams_list.txt'), 'w') as f:
            for mp in missing_pfams:
                f.write(f"{mp}\n")
        print(f"\nList saved to: {base_path}/missing_pfams_list.txt")
    else:
        print("\nAll required Pfam TSVs are present in your folder!")

# Run the scanner
scan_for_missing_pfams()

Mounted at /content/drive
Scanning folder: /content/drive/My Drive/Lauren Paprocki/PFAM TSV...

SCAN COMPLETE
Total Unique Pfams Required: 71
Total Files Found in Drive: 66
MISSING PFAMS: 10

Please download the TSV files for the following IDs:
 - Not Found
 - P25325
 - PF00383
 - PF00588
 - PF01189
 - PF01206
 - PF02403
 - PF04077
 - PF04358
 - PF13532

List saved to: /content/drive/My Drive/Lauren Paprocki/missing_pfams_list.txt


code to open seed file

In [ ]:
!pip install biopython
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

input_file = "PF00258.alignment.seed"
output_file = "PF00258_sequences.fasta"

clean_records = []

with open(input_file, "r") as in_handle:
    for record in SeqIO.parse(in_handle, "stockholm"):
        # Convert to string to safely remove gaps, then back to a Seq object
        sequence_string = str(record.seq).replace("-", "").replace(".", "")

        # Create a new record without the gaps
        new_record = SeqRecord(
            Seq(sequence_string),
            id=record.id,
            description=record.description
        )
        clean_records.append(new_record)

# Write all cleaned records to a FASTA file
with open(output_file, "w") as out_handle:
    SeqIO.write(clean_records, out_handle, "fasta")

print(f"Success! Created {output_file} with {len(clean_records)} sequences.")

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

Code fetching FASTA sequences from Uniprot IDs. you need to create a sheet that copies the column of uniprot + pfam id of the enzyme of interest. from here add the sheet id and it will save all the sequences to a fasta file.

In [ ]:
pip install pandas

In [ ]:
import requests
import time
import pandas as pd
import io

def get_fasta_sequences():
    # 1. Google Sheet Details
    sheet_id = "10WqxytaNMbxNGKmlt0l8m2xHotCo99pivbAwkVe0e9w"
    csv_url = f"https://docs.google.com/spreadsheets/d/10WqxytaNMbxNGKmlt0l8m2xHotCo99pivbAwkVe0e9w/export?format=csv"

    print("Fetching data from Google Sheets...")

    try:
        # Read the sheet. header=None because we are treating everything as data.
        df = pd.read_csv(csv_url, header=None)

        uniprot_ids = []

        # Iterate through every row in the first column (Column A)
        for val in df[0].astype(str):
            # Check if the cell contains a comma
            if ',' in val:
                # Split by comma and take the second part
                # We use .strip() to remove any hidden spaces or newline characters
                parts = val.split(',')
                if len(parts) >= 2:
                    uid = parts[1].strip()
                    # Just in case the next "PF" is stuck to the end of the ID
                    # e.g., "A0A650CYT1PF01926" -> we only want the ID
                    if "PF" in uid:
                        uid = uid.split("PF")[0].strip()

                    if uid:
                        uniprot_ids.append(uid)
            else:
                # If there's no comma, we check if the whole cell is an ID
                clean_val = val.strip()
                if clean_val and clean_val.lower() != 'nan':
                    uniprot_ids.append(clean_val)

        # Remove duplicates while keeping order
        uniprot_ids = list(dict.fromkeys(uniprot_ids))

    except Exception as e:
        print(f"Error reading Google Sheet: {e}")
        return

    if not uniprot_ids:
        print("No UniProt IDs found. Check if Column A contains the data.")
        return

    output_file = "sequences.fasta"
    base_url = "https://rest.uniprot.org/uniprotkb/"

    print(f"Starting retrieval of {len(uniprot_ids)} sequences...")

    with open(output_file, "w") as f:
        for i, uid in enumerate(uniprot_ids):
            try:
                response = requests.get(f"{base_url}{uid}.fasta")
                if response.status_code == 200:
                    f.write(response.text + "\n")
                    print(f"[{i+1}/{len(uniprot_ids)}] Success: {uid}")
                else:
                    print(f"[{i+1}/{len(uniprot_ids)}] Failed: {uid} (Status {response.status_code})")

                # UniProt is usually okay with 0.1s, but if you get 429 errors, increase this to 0.5
                time.sleep(0.1)

            except Exception as e:
                print(f"Error fetching {uid}: {e}")

    print(f"\nDone! All sequences saved to {output_file}")

if __name__ == "__main__":
    get_fasta_sequences()

Streaming output truncated to the last 5000 lines.
[3307/17652] Success: A0A1F5Z3A4
[3308/17652] Success: A0A1F5Z0F0
[3309/17652] Success: A0A1F5ZQP8
[3310/17652] Success: A0A1F5ZK23
[3311/17652] Success: A0A1F6A438
[3312/17652] Success: A0A1F6ACS5
[3313/17652] Success: A0A1F6AR07
[3314/17652] Success: A0A1F6BIR1
[3315/17652] Success: A0A1F6AHX3
[3316/17652] Success: A0A1F6AYQ1
[3317/17652] Success: A0A1F6AZF7
[3318/17652] Success: A0A1F6AV38
[3319/17652] Success: A0A1F6B0N6
[3320/17652] Success: A0A1F6B768
[3321/17652] Success: A0A1F6BET4
[3322/17652] Success: A0A143WR93;
[3323/17652] Success: A0A6V8QBP6;
[3324/17652] Success: V4J3A8
[3325/17652] Success: A0A9D2B5Z4;
[3326/17652] Success: A0A249DXV2;
[3327/17652] Success: A0A2M6WKM2
[3328/17652] Success: A0A2H0USK6
[3329/17652] Success: A0A2M6WHT7
[3330/17652] Success: A0A2H0UQL5
[3331/17652] Success: A0A2H0UQD3
[3332/17652] Success: A0A2H0ULI0
[3333/17652] Success: A0A1G1ZIN3
[3334/17652] Success: A0A1G1ZJ98
[3335/17652] Success: A0A

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

This parses through each sheet and determines if a pathway is full in each domain

In [ ]:
import pandas as pd
import os
import glob
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

def analyze_strict_pathways_exact_names():
    # --- CONFIGURATION ---
    target_folder_name = "Lauren Paprocki"

    domains = {
        "Archaea": "Archaea_Pathway_Analysis_Outputs",
        "Bacteria": "Bacterial_Pathway_Analysis_Outputs",
        "Eukaryotes": "Eukaryotic_Pathway_Analysis_Outputs",
        "Viruses": "Viral_Pathway_Analysis_Outputs"
    }

    # 1. Find the Root Folder
    root_search = '/content/drive/My Drive'
    found_root = None
    for root, dirs, files in os.walk(root_search):
        if target_folder_name in dirs:
            found_root = os.path.join(root, target_folder_name)
            break

    if not found_root:
        print(f"❌ Error: Could not find '{target_folder_name}'.")
        return

    # --- STEP 2: PRE-SCAN FOR MAXIMUM STEPS ---
    # This identifies the "Full" requirement for every unique pathway name found
    required_step_counts = {}

    all_files = []
    for sub_name in domains.values():
        for root, dirs, files in os.walk(found_root):
            if sub_name in dirs:
                all_files.extend(glob.glob(os.path.join(root, sub_name, "*.xlsx")))

    print("Pre-scanning headers to establish strict step requirements...")
    for file_path in all_files:
        try:
            # We only read headers to save memory/time
            hdr = pd.read_excel(file_path, nrows=0).columns
            step_headers = [col for col in hdr if ' Step ' in col]
            for h in step_headers:
                # We split but keep the name EXACTLY as it appears
                base_name = h.split(' Step ')[0] # Preserves spaces/parentheses exactly
                step_num = int(h.split(' Step ')[1])
                if base_name not in required_step_counts or step_num > required_step_counts[base_name]:
                    required_step_counts[base_name] = step_num
        except:
            continue

    # --- STEP 3: PERFORM ANALYSIS ---
    master_results = {}

    for domain_label, sub_name in domains.items():
        target_sub_path = None
        for root, dirs, files in os.walk(found_root):
            if sub_name in dirs:
                target_sub_path = os.path.join(root, sub_name)
                break

        if not target_sub_path:
            continue

        excel_files = glob.glob(os.path.join(target_sub_path, "*.xlsx"))
        print(f"Analyzing {len(excel_files)} files in {domain_label}...")

        for file_path in excel_files:
            file_name = os.path.basename(file_path)
            pfam_id = file_name.split('_')[-1].replace('.xlsx', '')

            try:
                df = pd.read_excel(file_path, engine='openpyxl')
                if 'Status' in df.columns and df.shape[1] < 3:
                    continue

                step_headers = [col for col in df.columns if ' Step ' in col]
                # Identify every pathway base name in this specific file
                pathways_in_sheet = set([h.split(' Step ')[0] for h in step_headers])

                for path_name in pathways_in_sheet:
                    if path_name not in master_results:
                        # Initialize with exact name
                        master_results[path_name] = {d: {"Full": "No", "Source": "N/A"} for d in domains.keys()}

                    # Get all columns belonging to this exact pathway name
                    path_cols = [h for h in step_headers if h.startswith(path_name)]
                    actual_step_count_in_file = len(path_cols)
                    required_count = required_step_counts.get(path_name, 0)

                    # STRICT CHECK: File must have ALL steps, and row must have NO 'missing' values
                    if actual_step_count_in_file >= required_count and required_count > 0:
                        data_clean = df[path_cols].astype(str).apply(lambda x: x.str.lower().str.strip())
                        is_full_row = (data_clean != "missing").all(axis=1)

                        if is_full_row.any():
                            if master_results[path_name][domain_label]["Full"] == "No":
                                master_results[path_name][domain_label] = {
                                    "Full": "Yes",
                                    "Source": pfam_id
                                }
            except Exception as e:
                pass

    # --- STEP 4: COMPILE FINAL SUMMARY ---
    rows = []
    # We sort the exact names alphabetically for the final sheet
    for path_name in sorted(master_results.keys()):
        entry = {"Pathway Name": path_name}
        for d_label in domains.keys():
            entry[f"{d_label} Full"] = master_results[path_name][d_label]["Full"]
            entry[f"{d_label} Source Pfam"] = master_results[path_name][d_label]["Source"]
        rows.append(entry)

    if rows:
        final_df = pd.DataFrame(rows)
        # Ensuring names are not truncated or altered during export
        output_path = "/content/drive/My Drive/STRICT_EXACT_NAME_SUMMARY.xlsx"
        final_df.to_excel(output_path, index=False)
        print(f"\n🚀 SUCCESS! Strict summary with {len(final_df)} pathways created.")
        print(f"Saved to: {output_path}")
    else:
        print("❌ No valid pathway data found.")

analyze_strict_pathways_exact_names()

Mounted at /content/drive
Pre-scanning headers to establish strict step requirements...
Analyzing 81 files in Archaea...
Analyzing 81 files in Bacteria...
Analyzing 81 files in Eukaryotes...
Analyzing 81 files in Viruses...

🚀 SUCCESS! Strict summary with 29 pathways created.
Saved to: /content/drive/My Drive/STRICT_EXACT_NAME_SUMMARY.xlsx


there was some missing pathways so this code corrects for that

In [ ]:
import pandas as pd
import os
import glob
import time
from google.colab import drive, auth
import gspread
from google.auth import default

# 1. Authenticate and Mount
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
creds, _ = default()
gc = gspread.authorize(creds)

def call_with_retry(func, *args, retries=5, delay=2, **kwargs):
    """Handles API 500 errors using exponential backoff."""
    for i in range(retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if i == retries - 1:
                raise e
            print(f"  [RETRY] API encounter. Waiting {delay}s...")
            time.sleep(delay)
            delay *= 2

def generate_targeted_missing_analysis():
    # --- CONFIGURATION ---
    spreadsheet_id = "16wAQWIvr0gSxje3iRoymMK9_7MdtavEeISlR6LNIDwg"
    base_path = '/content/drive/My Drive/Lauren Paprocki'
    pfam_folder = os.path.join(base_path, 'PFAM TSV')

    # NEW FOLDER FOR MISSING DATA
    output_base = os.path.join(base_path, 'Missing_Pathway_Data')

    # Define the specific pathways to catch
    missing_targets = [
        "tRNA-uridine34 modifications (B. subtilis)",
        "tRNA-uridine34 modifications (E. coli)",
        "tRNA-uridine34 modifications (eukaryotic cytoplasm)",
        "tRNA-uridine34 modifications (mammalian mitochondria)",
        "tRNA wobble base lysidine biosynthesis",
        "tRNA dihydrouridine",
        "tRNA 5-methyldihydrouridine biosynthesis"
    ]

    domains = {
        "Viral": "Viruses",
        "Bacterial": "Bacteria",
        "Archaeal": "Archaea",
        "Eukaryotic": "Eukaryota"
    }

    if not os.path.exists(output_base):
        os.makedirs(output_base)

    # 2. Load Sheet Data
    try:
        print("Connecting to Google Sheets...")
        spreadsheet = call_with_retry(gc.open_by_key, spreadsheet_id)
        enzyme_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("enzyme info").get_all_records))
        pathway_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("pathway data").get_all_records))
        gene_mapping_df = pd.DataFrame(call_with_retry(spreadsheet.worksheet("Gene pfam").get_all_records))

        for df in [enzyme_df, pathway_df, gene_mapping_df]:
            df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"CRITICAL: API Failure: {e}")
        return

    # 3. Mappings
    gene_to_metadata = {str(row['Gene']).strip().upper(): str(row['pfam ID']).strip() for _, row in gene_mapping_df.iterrows()}

    path_to_steps = {}
    for _, row in pathway_df.iterrows():
        p_name = str(row['Pathway Name']).strip()
        if p_name in missing_targets:
            steps = []
            gene_cols = [c for c in row.index if 'Gene' in c]
            for col in gene_cols:
                if pd.notna(row[col]) and str(row[col]).strip() != "":
                    step_genes = [g.strip() for g in str(row[col]).replace(';', ',').split(',')]
                    steps.append(step_genes)
            path_to_steps[p_name] = steps

    # 4. File Indexing
    all_tsvs = glob.glob(os.path.join(pfam_folder, "*.tsv"))
    pfam_file_lookup = {os.path.basename(f).split('_')[1].strip(): f for f in all_tsvs if len(os.path.basename(f).split('_')) > 1}
    pfam_data_cache = {}

    def get_pfam_data(pfam_id):
        if pfam_id in pfam_data_cache: return pfam_data_cache[pfam_id]
        path = pfam_file_lookup.get(pfam_id)
        org_map = {}
        if path:
            try:
                df = pd.read_csv(path, sep='\t', usecols=['Organism', 'Entry'], low_memory=False)
                for _, row in df.iterrows():
                    org_map[str(row['Organism']).strip().lower()] = str(row['Entry']).strip()
            except: pass
        pfam_data_cache[pfam_id] = org_map
        return org_map

    # 5. Targeted Processing
    # Filter enzyme_df for ONLY the missing pathways
    filtered_enzymes = enzyme_df[enzyme_df['Pathway Name'].apply(lambda x: any(target in str(x) for target in missing_targets))]

    print(f"Processing {len(filtered_enzymes)} enzymes for targeted missing pathways...")

    for domain_prefix, taxon_keyword in domains.items():
        domain_folder = os.path.join(output_base, f"{domain_prefix}_Missing_Outputs")
        if not os.path.exists(domain_folder): os.makedirs(domain_folder)

        for idx, target_row in filtered_enzymes.iterrows():
            current_pfam = str(target_row['pfam ID']).strip()
            pathway_list = [p.strip() for p in str(target_row['Pathway Name']).replace(';', ',').split(',') if p.strip() in missing_targets]

            if not pathway_list: continue

            primary_path = pfam_file_lookup.get(current_pfam)
            if not primary_path: continue

            # Load primary TSV and filter for domain
            primary_df = pd.read_csv(primary_path, sep='\t', low_memory=False)
            lineage_col = 'Taxonomic lineage' if 'Taxonomic lineage' in primary_df.columns else 'Organism'
            mask = primary_df[lineage_col].str.contains(taxon_keyword, case=False, na=False)
            unique_orgs = sorted(primary_df[mask]['Organism'].unique())

            if not unique_orgs: continue

            results = []
            for org_name in unique_orgs:
                org_lower = org_name.strip().lower()
                row_data = {"Organism": org_name}
                for p_name in pathway_list:
                    steps = path_to_steps.get(p_name, [])
                    for i, step_genes in enumerate(steps, 1):
                        col_header = f"{p_name} Step {i}"
                        found = []
                        for gene in step_genes:
                            pid = gene_to_metadata.get(gene.upper())
                            if pid:
                                lookup = get_pfam_data(pid)
                                if org_lower in lookup: found.append(f"{pid}, {lookup[org_lower]}")
                        row_data[col_header] = "; ".join(list(set(found))) if found else "missing"
                results.append(row_data)

            if results:
                out_file = os.path.join(domain_folder, f'{domain_prefix}_Targeted_{current_pfam}.xlsx')
                pd.DataFrame(results).to_excel(out_file, index=False)

    print(f"\n✅ All missing pathway data has been generated in: {output_base}")

generate_targeted_missing_analysis()

Mounted at /content/drive
Connecting to Google Sheets...
Processing 24 enzymes for targeted missing pathways...

✅ All missing pathway data has been generated in: /content/drive/My Drive/Lauren Paprocki/Missing_Pathway_Data
